# Multi-head latent attention (MLA) example

In [1]:
import torch
import time

In [40]:
torch.manual_seed(0)
torch.set_default_dtype(torch.float64)

n_embd, n_head, head_size = 128, 6, 64
d_latent = 32
T = 10

In [41]:
def rand(*shape):
    return torch.randn(*shape) * 0.05

In [42]:
# Normal Attention
W_K = [rand(n_embd, head_size) for _ in range(n_head)]
W_V = [rand(n_embd, head_size) for _ in range(n_head)]

W_DKV = rand(n_embd, d_latent)

W_UK = [rand(d_latent, head_size) for _ in range(n_head)]
W_UV = [rand(d_latent, head_size) for _ in range(n_head)]

W_Q = [rand(n_embd, head_size) for _ in range(n_head)]
W_O = rand(n_head * head_size, n_embd)

X = torch.randn(T, n_embd)
x_new = torch.randn(1, n_embd)


In [ ]:
print(f"W_O shape: {W_O.shape}")
print(f"W_K[1] shape: {W_K[0].shape}")
print(X.shape)

W_O shape: torch.Size([384, 128])
W_H1 shape: torch.Size([128, 64])
torch.Size([10, 128])


In [39]:
K_cache = [X @ W_K[h] for h in range(n_head)]
V_cache = [X @ W_V[h] for h in range(n_head)]

normal_per_token = 2 * n_head * head_size

C_KV = X @ W_DKV
print(f"   normal: {n_head} heads x (K {head_size} + V {head_size}) = {normal_per_token} numbers per token")
print(f"   MLA:    one C_KV of {d_latent} numbers per token")
print(f"   -> {normal_per_token / d_latent:.0f}x smaller cache\n")

   normal: 6 heads x (K 64 + V 64) = 768 numbers per token
   MLA:    one C_KV of 32 numbers per token
   -> 24x smaller cache



In [38]:
for h in range(2):
    k_h = C_KV @ W_UK[h]
    v_h = C_KV @ W_UK[h]
    print(f"   head {h}: keys {tuple(k_h.shape)}, values {tuple(v_h.shape)}, both unpacked from the same C_KV {tuple(C_KV.shape)}")

   head 0: keys (10, 64), values (10, 64), both unpacked from the same C_KV (10, 32)
   head 1: keys (10, 64), values (10, 64), both unpacked from the same C_KV (10, 32)
